This is our multi step algorithm to determine best locations for a coffee shop given our knowledge graph

1. Build a Node Regression Pipeline: 
    - Target Output: Avg_rating of a BusinessLocation Node
    - Input Features: Block Group attributes, Business Location Node locations, BusLoc - BusLoc relationships)

2. Extract Weights from Node Regression

3. Generate an "Optimal" Block Group

4. Similarity Search to Find Real Block Groups Closest to the Optimal Block Group
    - Rank by closeness in vector space 

5. Generate 10 New Sample Locations to Test
    - Use the geographical distribution of the businesses and county boundaries 
    - Test against zone locations to auto disqualify

6. Input the Generated Sample Locations into the Knowledge Graph

7. Use Node Regression Pipeline to Return Avg_Rating
    - Rank locations based on rating and other criteria


### 1. Setup

In [ ]:
import pandas as pd

# import geopandas as gpd

import os

from dotenv import load_dotenv
from decimal import Decimal
from neo4j import GraphDatabase


In [ ]:

load_dotenv("../.env")


class Config:
    def __init__(self, mode="LOCAL"):
        mode = mode.upper()

        if mode == "LOCAL":
            self.URI = os.getenv("NEO4J_URI_LOCAL")
            self.USER = os.getenv("NEO4J_USER_LOCAL")
            self.PASSWORD = os.getenv("NEO4J_PASSWORD_LOCAL")
        elif mode == "GROUP":
            self.URI = os.getenv("NEO4J_GROUP_URI")
            self.USER = os.getenv("NEO4J_GROUP_USER")
            self.PASSWORD = os.getenv("NEO4J_GROUP_PASSWORD")
        else:
            raise ValueError("Mode must be 'LOCAL' or 'GROUP'.")

        self.DATABASE = "neo4j"

config = Config(mode="LOCAL")
driver = GraphDatabase.driver(config.URI, auth=(config.USER, config.PASSWORD))

with driver.session(database=config.DATABASE) as session:
    session.run("MATCH (n) RETURN n LIMIT 1")
    print(f"Connection successful")

def query_neo4j(cypher_query: str, parameters: dict = None):
    with driver.session(database=config.DATABASE) as session:
        result = session.run(cypher_query, parameters)
        return result.data()

### 2. Best Location Algorithm

#### 2.1 Node Regression

#### 2.5 Sample Generation